# ResNet18 GeM + SupCon Binary 2-Class


## 1. Import libraries

Load PyTorch, torchvision, metrics, plotting, and utility libraries.


In [ ]:
import os
import json
import math
import random
import time
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler

import torchvision
from torchvision import transforms, models
from torchvision.utils import make_grid

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)

print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Config

Set the experiment name, output folders, seed, batch size, optimizer, and fine-tuning defaults.


In [ ]:
# Fill this manually if auto-detection does not pick the correct Kaggle dataset folder.
# Example: DATASET_ROOT = "/kaggle/input/your-dataset/celeba_spoof_subset_20000_micro_balanced"
DATASET_ROOT = ""

MODEL_KEY = "resnet18"
MODEL_NAME = "ResNet18 GeM SupCon 1View Binary"
EXPERIMENT_NAME = "resnet18_gem_supcon"
OUTPUT_DIR = Path("/kaggle/working/outputs/resnet18_gem_supcon")
USE_PRETRAINED = True

LABEL_MAPPING = {
    0: "Live",
    1: "Spoof",
}


def choose_batch_size(default_batch_size, fallback_batch_size, min_gpu_gb):
    if not torch.cuda.is_available():
        return fallback_batch_size
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if total_gb >= min_gpu_gb:
        return default_batch_size
    return fallback_batch_size


CONFIG = {
    "model_key": MODEL_KEY,
    "model_name": MODEL_NAME,
    "experiment_name": EXPERIMENT_NAME,
    "dataset_root": DATASET_ROOT,
    "output_dir": str(OUTPUT_DIR),
    "seed": 42,
    "image_size": 224,
    "default_batch_size": 64,
    "fallback_batch_size": 32,
    "batch_size": choose_batch_size(64, 32, 8),
    "num_workers": min(2, os.cpu_count() or 1),
    "num_epochs": 30,
    "freeze_epochs": 2,
    "early_stopping_patience": 5,
    "min_delta": 1e-5,
    "backbone_lr": 3e-05,
    "head_lr": 0.0003,
    "weight_decay": 1e-4,
    "use_pretrained": USE_PRETRAINED,
    "use_amp": torch.cuda.is_available(),
    "scheduler": "CosineAnnealingLR",
    "monitor": "val_acer",
    "use_gem": True,
    "use_cbam": False,
    "use_supcon": True,
    "supcon_views": 1,
    "supcon_weight": 0.05,
    "supcon_temperature": 0.07,
    "projection_dim": 128,
}

DIRS = {
    "checkpoints": OUTPUT_DIR / "checkpoints",
    "metrics": OUTPUT_DIR / "metrics",
    "curves": OUTPUT_DIR / "curves",
    "confusion_matrix": OUTPUT_DIR / "confusion_matrix",
    "predictions": OUTPUT_DIR / "predictions",
    "samples": OUTPUT_DIR / "samples",
    "config": OUTPUT_DIR / "config",
    "logs": OUTPUT_DIR / "logs",
}
for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

LOG_PATH = DIRS["logs"] / "train_log.txt"
if LOG_PATH.exists():
    LOG_PATH.unlink()


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def json_default(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    return str(obj)


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=json_default)


def log_message(message):
    print(message)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(str(message) + "\n")


set_seed(CONFIG["seed"])
save_json(CONFIG, DIRS["config"] / "config.json")
save_json({"0": "Live", "1": "Spoof", "Live": 0, "Spoof": 1}, DIRS["config"] / "label_mapping.json")

print(json.dumps(CONFIG, indent=2, default=json_default))


## 3. Find dataset root and load CSV

Use `DATASET_ROOT` if provided; otherwise find a valid dataset folder under `/kaggle/input`. The notebook always reads the provided `train.csv`, `val.csv`, and `test.csv` without creating a new split.


In [ ]:
def validate_dataset_root(root):
    root = Path(root)
    required = ["train.csv", "val.csv", "test.csv", "metadata_all.csv"]
    missing = [name for name in required if not (root / name).exists()]
    if not (root / "images").exists():
        missing.append("images/")
    return len(missing) == 0, missing


def find_dataset_root(user_root=""):
    if str(user_root).strip():
        root = Path(user_root).expanduser()
        ok, missing = validate_dataset_root(root)
        if not ok:
            raise FileNotFoundError(
                f"DATASET_ROOT={root} is missing required items: {missing}. "
                "Point DATASET_ROOT to the folder that contains train.csv, val.csv, test.csv, metadata_all.csv and images/."
            )
        return root

    search_root = Path("/kaggle/input")
    candidates = []
    if search_root.exists():
        for train_csv in search_root.rglob("train.csv"):
            root = train_csv.parent
            ok, _ = validate_dataset_root(root)
            if ok:
                candidates.append(root)

    if not candidates:
        raise FileNotFoundError(
            "Could not auto-detect dataset root under /kaggle/input. "
            "Please set DATASET_ROOT in the Config cell."
        )

    candidates = sorted(
        candidates,
        key=lambda p: ((p / "metadata_all.csv").stat().st_size, -len(str(p))),
        reverse=True,
    )
    print("Candidate dataset roots:")
    for candidate in candidates:
        print(" -", candidate)
    print("Using dataset root:", candidates[0])
    return candidates[0]


dataset_root = find_dataset_root(DATASET_ROOT)
CONFIG["dataset_root"] = str(dataset_root)
save_json(CONFIG, DIRS["config"] / "config.json")

train_df = pd.read_csv(dataset_root / "train.csv")
val_df = pd.read_csv(dataset_root / "val.csv")
test_df = pd.read_csv(dataset_root / "test.csv")
metadata_df = pd.read_csv(dataset_root / "metadata_all.csv")

required_columns = [
    "cropped_path",
    "binary_label",
    "class_name",
    "spoof_type_name",
    "micro_type_id",
    "attack_group",
    "subject_id",
    "split",
]

for split_name, df in {"train": train_df, "val": val_df, "test": test_df}.items():
    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{split_name}.csv is missing required columns: {missing_cols}")
    labels = set(df["binary_label"].dropna().astype(int).unique().tolist())
    if not labels.issubset({0, 1}):
        raise ValueError(f"{split_name}.csv has unexpected binary_label values: {labels}")
    if "split" in df.columns:
        found_splits = set(df["split"].dropna().astype(str).unique().tolist())
        if found_splits != {split_name}:
            raise ValueError(f"{split_name}.csv split column should contain only '{split_name}', found {found_splits}")

print("Dataset root:", dataset_root)
print("Rows:", {"train": len(train_df), "val": len(val_df), "test": len(test_df), "metadata_all": len(metadata_df)})
display(train_df.head())


## 4. Resolve image paths

Old `/kaggle/working/...` paths are remapped by taking the relative path after `images/` and joining it with the current dataset root.


In [ ]:
split_dfs = {"train": train_df, "val": val_df, "test": test_df}

print("Number of images by split")
for split_name, df in split_dfs.items():
    print(f"{split_name}: {len(df):,}")

print("\nLive/Spoof distribution by split")
binary_dist = pd.concat(split_dfs, names=["csv_split"]).reset_index(level=0).rename(columns={"level_0": "csv_split"})
display(pd.crosstab(binary_dist["csv_split"], binary_dist["class_name"]))
display(pd.crosstab(binary_dist["csv_split"], binary_dist["binary_label"]))

print("\nMicro spoof type distribution by split")
display(pd.crosstab(binary_dist["csv_split"], binary_dist["micro_type_id"]))

print("\nSpoof type distribution by split")
display(pd.crosstab(binary_dist["csv_split"], binary_dist["spoof_type_name"]))

print("\nAttack group distribution by split")
display(pd.crosstab(binary_dist["csv_split"], binary_dist["attack_group"]))


def check_subject_overlap(split_dfs):
    if any("subject_id" not in df.columns for df in split_dfs.values()):
        print("subject_id column is missing in at least one split; subject overlap check skipped.")
        return

    subject_sets = {
        split_name: set(df["subject_id"].astype(str).tolist())
        for split_name, df in split_dfs.items()
    }
    overlaps = {}
    names = list(subject_sets.keys())
    for i, left in enumerate(names):
        for right in names[i + 1:]:
            overlap = subject_sets[left] & subject_sets[right]
            overlaps[f"{left}_{right}"] = len(overlap)
            if overlap:
                raise AssertionError(
                    f"Subject overlap detected between {left} and {right}: "
                    f"{len(overlap)} subjects. Examples: {list(sorted(overlap))[:10]}"
                )
    print("Subject overlap check passed:", overlaps)


check_subject_overlap(split_dfs)

subject_summary_path = dataset_root / "subject_summary.csv"
if subject_summary_path.exists():
    subject_summary_df = pd.read_csv(subject_summary_path)
    print("\nsubject_summary.csv rows:", len(subject_summary_df))
    display(subject_summary_df.head())
    if "has_both" in subject_summary_df.columns:
        bad_subjects = subject_summary_df[subject_summary_df["has_both"].astype(str).str.lower() != "true"]
        if len(bad_subjects) > 0:
            raise AssertionError(f"Found subjects without both live and spoof samples: {len(bad_subjects)}")
        print("Every subject in subject_summary.csv has both Live and Spoof samples.")
else:
    print("subject_summary.csv not found; skipping per-subject live/spoof check.")


In [ ]:
def relative_after_images(path_value):
    if pd.isna(path_value):
        return None
    text = str(path_value).strip().replace("\\", "/")
    marker = "images/"
    idx = text.find(marker)
    if idx >= 0:
        return text[idx + len(marker):].lstrip("/")
    if text.startswith(marker):
        return text[len(marker):].lstrip("/")
    return None


def safe_path_exists(path_value):
    try:
        return Path(str(path_value)).exists()
    except (OSError, ValueError):
        return False


def resolve_image_path(row, root):
    cropped_path = row.get("cropped_path", "")
    if isinstance(cropped_path, str) and cropped_path.strip() and safe_path_exists(cropped_path):
        return str(Path(cropped_path))

    rel = relative_after_images(cropped_path)
    if rel:
        candidate = root / "images" / rel
        return str(candidate)

    # Fallback for unusual CSVs: reconstruct from split/class/spoof_type and file name.
    filename = Path(str(cropped_path).replace("\\", "/")).name
    split = str(row.get("split", "")).strip()
    class_name = str(row.get("class_name", row.get("folder_label", ""))).strip().lower()
    spoof_type = str(row.get("spoof_type_name", "")).strip().replace(" ", "_")
    if filename and split:
        if class_name == "live":
            candidate = root / "images" / split / "live" / filename
        else:
            candidate = root / "images" / split / "spoof" / spoof_type / filename
        return str(candidate)

    return str(cropped_path)


def attach_resolved_paths(df, split_name, root):
    df = df.copy()
    df["image_path_resolved"] = df.apply(lambda row: resolve_image_path(row, root), axis=1)
    missing_mask = ~df["image_path_resolved"].map(lambda p: Path(p).exists())
    missing_count = int(missing_mask.sum())
    print(f"{split_name}: missing image paths = {missing_count}/{len(df)}")
    if missing_count > 0:
        display(df.loc[missing_mask, ["cropped_path", "image_path_resolved", "split", "class_name", "spoof_type_name"]].head(20))
        raise FileNotFoundError(
            f"{split_name} still has {missing_count} missing images after path resolution. "
            "Check DATASET_ROOT and the images/ folder structure."
        )
    return df


train_df = attach_resolved_paths(train_df, "train", dataset_root)
val_df = attach_resolved_paths(val_df, "val", dataset_root)
test_df = attach_resolved_paths(test_df, "test", dataset_root)

print("\nResolved path examples")
display(train_df[["cropped_path", "image_path_resolved", "binary_label", "class_name", "spoof_type_name"]].head())


## 5. Dataset and DataLoader

Create binary classification datasets and dataloaders with moderate texture-preserving augmentation.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG["image_size"], scale=(0.90, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08, hue=0.02),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class FaceAntiSpoofDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["image_path_resolved"]
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        label = int(row["binary_label"])
        return image, label, idx


train_dataset = FaceAntiSpoofDataset(train_df, train_transform)
val_dataset = FaceAntiSpoofDataset(val_df, eval_transform)
test_dataset = FaceAntiSpoofDataset(test_df, eval_transform)

loader_kwargs = {
    "batch_size": CONFIG["batch_size"],
    "num_workers": CONFIG["num_workers"],
    "pin_memory": torch.cuda.is_available(),
}
if CONFIG["num_workers"] > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(train_dataset, shuffle=True, drop_last=False, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, drop_last=False, **loader_kwargs)

print("DataLoader batch size:", CONFIG["batch_size"])
print("Train batches:", len(train_loader), "Val batches:", len(val_loader), "Test batches:", len(test_loader))


def denormalize_tensor_batch(batch):
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return torch.clamp(batch.cpu() * std + mean, 0, 1)


def show_transformed_samples(dataset, n=8):
    sample_loader = DataLoader(dataset, batch_size=n, shuffle=True, num_workers=0)
    images, labels, _ = next(iter(sample_loader))
    grid = make_grid(denormalize_tensor_batch(images), nrow=4)
    plt.figure(figsize=(12, 6))
    plt.imshow(grid.permute(1, 2, 0))
    title = " | ".join([LABEL_MAPPING[int(label)] for label in labels])
    plt.title(f"Train samples after transform: {title}")
    plt.axis("off")
    plt.show()


show_transformed_samples(train_dataset, n=8)


## 6. Model definition

Load the torchvision backbone, replace the final classifier with 2 classes, and use separate learning rates for backbone and classifier.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WEIGHTS_ENUM = {
    "resnet18": "ResNet18_Weights",
    "resnet50": "ResNet50_Weights",
    "efficientnet_b0": "EfficientNet_B0_Weights",
    "efficientnet_b3": "EfficientNet_B3_Weights",
}


def build_backbone(model_key, weights=None, pretrained_flag=None):
    model_fn = getattr(models, model_key)
    if pretrained_flag is not None:
        return model_fn(pretrained=pretrained_flag)
    return model_fn(weights=weights)


class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6, trainable=True):
        super().__init__()
        if trainable:
            self.p = nn.Parameter(torch.ones(1) * p)
        else:
            self.p = torch.tensor([p])
        self.eps = eps

    def forward(self, x):
        x = x.clamp(min=self.eps).pow(self.p)
        x = F.avg_pool2d(x, kernel_size=(x.size(-2), x.size(-1)))
        x = x.pow(1.0 / self.p)
        return x.flatten(1)


class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07, eps=1e-8):
        super().__init__()
        self.temperature = temperature
        self.eps = eps

    def forward(self, features, labels):
        if features.ndim != 2:
            raise ValueError(f"Expected features with shape [batch_size, dim], got {tuple(features.shape)}")

        device = features.device
        features = F.normalize(features.float(), dim=1)
        labels = labels.contiguous().view(-1, 1)
        batch_size = features.shape[0]

        mask = torch.eq(labels, labels.T).float().to(device)

        logits = torch.matmul(features, features.T) / self.temperature
        logits = logits - logits.max(dim=1, keepdim=True)[0].detach()

        logits_mask = torch.ones_like(mask) - torch.eye(batch_size, device=device)
        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + self.eps)

        pos_count = mask.sum(dim=1)
        valid = pos_count > 0
        if valid.sum() == 0:
            return features.sum() * 0.0

        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (pos_count + self.eps)
        loss = -mean_log_prob_pos[valid].mean()
        return loss

class ResNet18GeMSupCon(nn.Module):
    def __init__(self, base_model, num_classes=2, projection_dim=128):
        super().__init__()

        self.conv1 = base_model.conv1
        self.bn1 = base_model.bn1
        self.relu = base_model.relu
        self.maxpool = base_model.maxpool

        self.layer1 = base_model.layer1
        self.layer2 = base_model.layer2
        self.layer3 = base_model.layer3
        self.layer4 = base_model.layer4

        in_features = base_model.fc.in_features
        self.feature_dim = in_features
        self.avgpool = GeM()
        self.fc = nn.Linear(in_features, num_classes)
        self.projection_head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, projection_dim),
        )

    def forward_features(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        return x

    def forward(self, x, return_features=False):
        features = self.forward_features(x)
        logits = self.fc(features)
        if return_features:
            projection = self.projection_head(features)
            projection = F.normalize(projection, dim=1)
            return logits, projection
        return logits


def disable_inplace_modules(module):
    for child in module.modules():
        if hasattr(child, "inplace"):
            child.inplace = False


def create_model(model_key, num_classes=2, use_pretrained=True):
    pretrained_loaded = False
    base_model = None

    if use_pretrained:
        try:
            weights_cls = getattr(models, WEIGHTS_ENUM[model_key])
            weights = weights_cls.DEFAULT
            base_model = build_backbone(model_key, weights=weights)
            pretrained_loaded = True
            print(f"Loaded ImageNet pretrained weights: {WEIGHTS_ENUM[model_key]}.DEFAULT")
        except Exception as exc:
            print("WARNING: Could not load torchvision pretrained weights.")
            print("Reason:", repr(exc))
            print("Falling back to weights=None. Results will likely be weaker.")

    if base_model is None:
        try:
            base_model = build_backbone(model_key, weights=None)
        except TypeError:
            base_model = build_backbone(model_key, pretrained_flag=False)

    if model_key == "resnet18":
        model = ResNet18GeMSupCon(
            base_model,
            num_classes=num_classes,
            projection_dim=CONFIG["projection_dim"],
        )
        head = nn.ModuleList([model.fc, model.projection_head])
    else:
        raise ValueError(f"Unsupported model_key for this notebook: {model_key}")

    disable_inplace_modules(head)

    return model, head, pretrained_loaded


def set_backbone_trainable(model, head, trainable):
    for parameter in model.parameters():
        parameter.requires_grad = trainable
    for parameter in head.parameters():
        parameter.requires_grad = True


def make_optimizer(model, head):
    head_param_ids = {id(parameter) for parameter in head.parameters()}
    backbone_params = [parameter for parameter in model.parameters() if id(parameter) not in head_param_ids]
    head_params = list(head.parameters())
    optimizer = AdamW(
        [
            {"params": backbone_params, "lr": CONFIG["backbone_lr"], "name": "backbone"},
            {"params": head_params, "lr": CONFIG["head_lr"], "name": "classifier"},
        ],
        weight_decay=CONFIG["weight_decay"],
    )
    return optimizer


def count_parameters(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable


model, classifier_head, pretrained_loaded = create_model(
    CONFIG["model_key"],
    num_classes=2,
    use_pretrained=CONFIG["use_pretrained"],
)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
supcon_criterion = SupConLoss(temperature=CONFIG["supcon_temperature"])
optimizer = make_optimizer(model, classifier_head)
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["num_epochs"], eta_min=1e-6)
scaler = GradScaler(enabled=CONFIG["use_amp"])

if CONFIG["freeze_epochs"] > 0:
    set_backbone_trainable(model, classifier_head, trainable=False)
else:
    set_backbone_trainable(model, classifier_head, trainable=True)

total_params, trainable_params = count_parameters(model)
print(model.__class__.__name__)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters at start: {trainable_params:,}")
print("Pretrained loaded:", pretrained_loaded)

print("Pooling layer:", model.avgpool)
print("Classifier/projection heads:", classifier_head)
print("SupCon criterion:", supcon_criterion)

was_training = model.training
model.eval()
with torch.no_grad():
    dummy_input = torch.randn(2, 3, CONFIG["image_size"], CONFIG["image_size"], device=device)
    dummy_logits, dummy_embeddings = model(dummy_input, return_features=True)
print("Forward pass output shape:", tuple(dummy_logits.shape))
print("Projection embedding shape:", tuple(dummy_embeddings.shape))
assert tuple(dummy_logits.shape) == (2, 2)
assert tuple(dummy_embeddings.shape) == (2, CONFIG["projection_dim"])
if was_training:
    model.train()


In [ ]:
batch = next(iter(train_loader))
images, labels, indices = batch

images = images.to(device)
labels = labels.to(device)

was_training = model.training
model.eval()
with torch.no_grad():
    logits, embeddings = model(images, return_features=True)

print("images:", images.shape)
print("logits:", logits.shape)
print("embeddings:", embeddings.shape)

assert images.ndim == 4 and images.shape[1] == 3
assert logits.shape == (images.shape[0], 2)
assert embeddings.shape == (images.shape[0], CONFIG["projection_dim"])
assert not torch.isnan(supcon_criterion(embeddings, labels)).item()

if was_training:
    model.train()


## 7. Training and validation functions

Define one-epoch training, evaluation, metric computation, and prediction table helpers.


In [ ]:
def compute_fas_metrics(y_true, y_pred, y_prob=None):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

    apcer = fn / (tp + fn) if (tp + fn) > 0 else 0.0
    bpcer = fp / (tn + fp) if (tn + fp) > 0 else 0.0
    acer = (apcer + bpcer) / 2.0

    metrics = {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "apcer": float(apcer),
        "bpcer": float(bpcer),
        "acer": float(acer),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "roc_auc": None,
    }
    if y_prob is not None and len(np.unique(y_true)) == 2:
        try:
            metrics["roc_auc"] = float(roc_auc_score(y_true, np.asarray(y_prob)[:, 1]))
        except Exception as exc:
            print("ROC-AUC could not be computed:", repr(exc))
    return metrics


# Backward-compatible alias for older cells or ad-hoc reruns.
compute_binary_metrics = compute_fas_metrics


def build_predictions_df(source_df, indices, y_true, y_pred, y_prob):
    meta = source_df.iloc[np.asarray(indices).astype(int)].reset_index(drop=True).copy()
    output = pd.DataFrame({
        "image_path": meta["image_path_resolved"].astype(str),
        "subject_id": meta["subject_id"] if "subject_id" in meta.columns else "",
        "true_label": np.asarray(y_true).astype(int),
        "pred_label": np.asarray(y_pred).astype(int),
        "true_name": [LABEL_MAPPING[int(label)] for label in y_true],
        "pred_name": [LABEL_MAPPING[int(label)] for label in y_pred],
        "prob_live": np.asarray(y_prob)[:, 0],
        "prob_spoof": np.asarray(y_prob)[:, 1],
    })
    output["confidence"] = output[["prob_live", "prob_spoof"]].max(axis=1)
    output["is_correct"] = output["true_label"] == output["pred_label"]

    extra_columns = [
        "class_name",
        "spoof_type_name",
        "micro_type_id",
        "spoof_type_id",
        "attack_group",
        "attack_group_id",
        "split",
        "cropped_path",
        "rel_path",
    ]
    for column in extra_columns:
        if column in meta.columns:
            output[column] = meta[column].values
    return output



def train_one_epoch(model, loader, criterion, supcon_criterion, optimizer, scaler):
    model.train()
    running_loss = 0.0
    running_ce_loss = 0.0
    running_supcon_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    progress = tqdm(loader, desc="Train", leave=False)
    for images, labels, indices in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=CONFIG["use_amp"]):
            logits, embeddings = model(images, return_features=True)
            ce_loss = criterion(logits, labels)
            supcon_loss = supcon_criterion(embeddings, labels)
            loss = ce_loss + CONFIG["supcon_weight"] * supcon_loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        probs = torch.softmax(logits.detach(), dim=1)
        preds = torch.argmax(probs, dim=1)

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        running_ce_loss += ce_loss.item() * batch_size
        running_supcon_loss += supcon_loss.item() * batch_size
        all_labels.append(labels.detach().cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            ce=f"{ce_loss.item():.4f}",
            supcon=f"{supcon_loss.item():.4f}",
        )

    y_true = np.concatenate(all_labels)
    y_pred = np.concatenate(all_preds)
    y_prob = np.concatenate(all_probs)
    metrics = compute_fas_metrics(y_true, y_pred, y_prob)
    metrics["loss"] = float(running_loss / len(loader.dataset))
    metrics["ce_loss"] = float(running_ce_loss / len(loader.dataset))
    metrics["supcon_loss"] = float(running_supcon_loss / len(loader.dataset))
    return metrics


@torch.no_grad()
def evaluate(model, loader, criterion=None, source_df=None, desc="Eval"):
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []
    all_indices = []

    progress = tqdm(loader, desc=desc, leave=False)
    for images, labels, indices in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(enabled=CONFIG["use_amp"]):
            logits = model(images)
            loss = criterion(logits, labels) if criterion is not None else None

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        if loss is not None:
            running_loss += loss.item() * labels.size(0)
        all_labels.append(labels.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())
        all_probs.append(probs.detach().cpu().numpy())
        all_indices.extend(indices.detach().cpu().numpy().tolist())

    y_true = np.concatenate(all_labels)
    y_pred = np.concatenate(all_preds)
    y_prob = np.concatenate(all_probs)
    metrics = compute_fas_metrics(y_true, y_pred, y_prob)
    if criterion is not None:
        metrics["loss"] = float(running_loss / len(loader.dataset))

    predictions_df = None
    if source_df is not None:
        predictions_df = build_predictions_df(source_df, all_indices, y_true, y_pred, y_prob)
    return metrics, predictions_df


## 8. Training loop

Fine-tune with optional warm-up freezing, mixed precision on GPU, cosine scheduler, early stopping, and best checkpoint by validation F1.


In [ ]:
best_checkpoint_path = DIRS["checkpoints"] / "best_model.pth"
last_checkpoint_path = DIRS["checkpoints"] / "last_model.pth"
best_val_acer = float("inf")
best_val_apcer = float("inf")
selection_eps = 1e-6
epochs_without_improvement = 0
history = []

log_message(f"Started training at {datetime.now().isoformat(timespec='seconds')}")
log_message(f"Model: {CONFIG['model_name']}")
log_message(f"Selection metric: validation ACER, tie-breaker validation APCER")
log_message(f"SupCon weight: {CONFIG['supcon_weight']} | temperature: {CONFIG['supcon_temperature']}")
log_message(f"Output dir: {OUTPUT_DIR}")

for epoch in range(1, CONFIG["num_epochs"] + 1):
    epoch_start = time.time()

    if epoch == CONFIG["freeze_epochs"] + 1 and CONFIG["freeze_epochs"] > 0:
        set_backbone_trainable(model, classifier_head, trainable=True)
        _, trainable_params = count_parameters(model)
        log_message(f"Epoch {epoch}: unfroze backbone. Trainable parameters: {trainable_params:,}")

    train_metrics = train_one_epoch(model, train_loader, criterion, supcon_criterion, optimizer, scaler)
    val_metrics, _ = evaluate(model, val_loader, criterion, source_df=None, desc="Val")
    scheduler.step()

    current_lrs = {group.get("name", str(i)): group["lr"] for i, group in enumerate(optimizer.param_groups)}
    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_total_loss": train_metrics["loss"],
        "train_ce_loss": train_metrics["ce_loss"],
        "train_supcon_loss": train_metrics["supcon_loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_precision": train_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "train_f1": train_metrics["f1"],
        "train_apcer": train_metrics["apcer"],
        "train_bpcer": train_metrics["bpcer"],
        "train_acer": train_metrics["acer"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_apcer": val_metrics["apcer"],
        "val_bpcer": val_metrics["bpcer"],
        "val_acer": val_metrics["acer"],
        "val_roc_auc": val_metrics["roc_auc"],
        "backbone_lr": current_lrs.get("backbone"),
        "head_lr": current_lrs.get("classifier"),
        "elapsed_sec": time.time() - epoch_start,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(DIRS["metrics"] / "epoch_history.csv", index=False)

    log_message(
        f"Epoch {epoch:02d}/{CONFIG['num_epochs']} | "
        f"train_loss={row['train_loss']:.4f} "
        f"train_ce={row['train_ce_loss']:.4f} "
        f"train_supcon={row['train_supcon_loss']:.4f} | "
        f"val_loss={row['val_loss']:.4f} "
        f"val_acc={row['val_accuracy']:.4f} "
        f"val_precision={row['val_precision']:.4f} "
        f"val_recall={row['val_recall']:.4f} "
        f"val_f1={row['val_f1']:.4f} "
        f"val_apcer={row['val_apcer']:.2%} "
        f"val_bpcer={row['val_bpcer']:.2%} "
        f"val_acer={row['val_acer']:.2%}"
    )

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "config": CONFIG,
        "label_mapping": LABEL_MAPPING,
        "selection_metric": "val_acer",
        "best_val_acer": best_val_acer,
        "best_val_apcer": best_val_apcer,
        "pretrained_loaded": pretrained_loaded,
    }
    torch.save(checkpoint, last_checkpoint_path)

    improved = (
        val_metrics["acer"] < best_val_acer - selection_eps
        or (
            abs(val_metrics["acer"] - best_val_acer) <= selection_eps
            and val_metrics["apcer"] < best_val_apcer
        )
    )
    if improved:
        best_val_acer = val_metrics["acer"]
        best_val_apcer = val_metrics["apcer"]
        checkpoint["best_val_acer"] = best_val_acer
        checkpoint["best_val_apcer"] = best_val_apcer
        torch.save(checkpoint, best_checkpoint_path)
        epochs_without_improvement = 0
        log_message(
            f"  Saved new best checkpoint: "
            f"val_acer={best_val_acer:.4f}, val_apcer={best_val_apcer:.4f}"
        )
    else:
        epochs_without_improvement += 1
        log_message(f"  No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= CONFIG["early_stopping_patience"]:
        log_message("Early stopping triggered.")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(DIRS["metrics"] / "epoch_history.csv", index=False)
log_message(f"Finished training at {datetime.now().isoformat(timespec='seconds')}")
print("Best validation ACER:", best_val_acer)
print("Best validation APCER:", best_val_apcer)
print("Best checkpoint:", best_checkpoint_path)


## 9. Test evaluation

Reload the best checkpoint and evaluate validation and test splits.


In [ ]:
if not best_checkpoint_path.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {best_checkpoint_path}")

checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)

best_val_acer = float(checkpoint.get("best_val_acer", best_val_acer))
best_val_apcer = float(checkpoint.get("best_val_apcer", best_val_apcer))
val_metrics, val_predictions = evaluate(model, val_loader, criterion, source_df=val_df, desc="Val(best)")
test_metrics, test_predictions = evaluate(model, test_loader, criterion, source_df=test_df, desc="Test")

print("Validation metrics from best checkpoint")
print(json.dumps(val_metrics, indent=2, default=json_default))
print()
print("Test metrics")
print(json.dumps(test_metrics, indent=2, default=json_default))


## 10. Save outputs

Save metrics, predictions, classification report, and attack-type analysis tables.


In [ ]:
def with_context(metrics, split_name, n_samples):
    result = {
        "split": split_name,
        "num_samples": int(n_samples),
        "model_name": CONFIG["model_name"],
        "model_key": CONFIG["model_key"],
        "experiment_name": CONFIG["experiment_name"],
        "selection_metric": "val_acer",
        "best_val_acer": float(best_val_acer),
        "best_val_apcer": float(best_val_apcer),
    }
    result.update(dict(metrics))
    result.update({
        "checkpoint_path": str(best_checkpoint_path),
        "output_dir": str(OUTPUT_DIR),
        "dataset_root": str(dataset_root),
    })
    return result


val_metrics_to_save = with_context(val_metrics, "val", len(val_df))
test_metrics_to_save = with_context(test_metrics, "test", len(test_df))

save_json(val_metrics_to_save, DIRS["metrics"] / "val_metrics.json")
save_json(test_metrics_to_save, DIRS["metrics"] / "test_metrics.json")
save_json(test_metrics_to_save, DIRS["metrics"] / f"{EXPERIMENT_NAME}.json")

val_predictions.to_csv(DIRS["predictions"] / "val_predictions.csv", index=False)
test_predictions.to_csv(DIRS["predictions"] / "test_predictions.csv", index=False)

report = classification_report(
    test_predictions["true_label"],
    test_predictions["pred_label"],
    labels=[0, 1],
    target_names=["Live", "Spoof"],
    output_dict=True,
    zero_division=0,
)
classification_report_df = pd.DataFrame(report).transpose()
classification_report_df.to_csv(DIRS["metrics"] / "classification_report.csv")


def summarize_spoof_group(predictions, group_col):
    if group_col not in predictions.columns:
        return pd.DataFrame()
    work = predictions[predictions["true_label"] == 1].copy()
    if work.empty:
        return pd.DataFrame()

    rows = []
    for group_value, group in work.groupby(group_col, dropna=False):
        correct = int(group["is_correct"].sum())
        total = int(len(group))
        rows.append({
            group_col: group_value,
            "n": total,
            "correct": correct,
            "wrong": total - correct,
            "accuracy": float(correct / total) if total else None,
            "spoof_recall": float((group["pred_label"] == 1).mean()),
            "avg_prob_spoof": float(group["prob_spoof"].mean()),
        })
    return pd.DataFrame(rows).sort_values(["accuracy", "n"], ascending=[True, False]).reset_index(drop=True)


test_by_attack_group = summarize_spoof_group(test_predictions, "attack_group")
test_by_spoof_type = summarize_spoof_group(test_predictions, "spoof_type_name")

test_by_attack_group.to_csv(DIRS["metrics"] / "test_by_attack_group.csv", index=False)
test_by_spoof_type.to_csv(DIRS["metrics"] / "test_by_spoof_type.csv", index=False)

print("Saved metrics and predictions under:", OUTPUT_DIR)
display(classification_report_df)
print("Test by attack group")
display(test_by_attack_group)
print("Test by spoof type")
display(test_by_spoof_type)


## 11. Visualize results

Plot and save loss, accuracy, F1, confusion matrices, and ROC curve.


In [ ]:
history_df = pd.read_csv(DIRS["metrics"] / "epoch_history.csv")


CONFUSION_LABELS = [0, 1]
CONFUSION_DISPLAY_LABELS = ["Live", "Spoof"]


def plot_history_curve(history_df, train_col, val_col, ylabel, save_path):
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df[train_col], marker="o", label=train_col)
    plt.plot(history_df["epoch"], history_df[val_col], marker="o", label=val_col)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(f"{CONFIG['model_name']} - {ylabel}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=160)
    plt.show()


def plot_validation_fas_curve(history_df, save_path):
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["val_apcer"], marker="o", label="val_apcer")
    plt.plot(history_df["epoch"], history_df["val_bpcer"], marker="o", label="val_bpcer")
    plt.plot(history_df["epoch"], history_df["val_acer"], marker="o", label="val_acer")
    plt.xlabel("Epoch")
    plt.ylabel("Error rate")
    plt.title(f"{CONFIG['model_name']} - Validation FAS Metrics")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=160)
    plt.show()


plot_history_curve(history_df, "train_loss", "val_loss", "Loss", DIRS["curves"] / "loss_curve.png")
plot_history_curve(history_df, "train_accuracy", "val_accuracy", "Accuracy", DIRS["curves"] / "accuracy_curve.png")
plot_history_curve(history_df, "train_f1", "val_f1", "F1-score", DIRS["curves"] / "f1_curve.png")
plot_history_curve(history_df, "train_acer", "val_acer", "ACER", DIRS["curves"] / "acer_curve.png")
plot_validation_fas_curve(history_df, DIRS["curves"] / "fas_metrics_curve.png")


def plot_confusion(y_true, y_pred, normalize, save_path, title):
    cm = confusion_matrix(y_true, y_pred, labels=CONFUSION_LABELS, normalize="true" if normalize else None)
    plt.figure(figsize=(6, 5))
    fmt = ".2f" if normalize else "d"
    sns.heatmap(
        cm,
        annot=True,
        fmt=fmt,
        cmap="Blues",
        xticklabels=CONFUSION_DISPLAY_LABELS,
        yticklabels=CONFUSION_DISPLAY_LABELS,
        cbar=False,
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=160)
    plt.show()


plot_confusion(
    test_predictions["true_label"],
    test_predictions["pred_label"],
    normalize=False,
    save_path=DIRS["confusion_matrix"] / "confusion_matrix_raw.png",
    title=f"{CONFIG['model_name']} - Confusion Matrix",
)
plot_confusion(
    test_predictions["true_label"],
    test_predictions["pred_label"],
    normalize=True,
    save_path=DIRS["confusion_matrix"] / "confusion_matrix_normalized.png",
    title=f"{CONFIG['model_name']} - Normalized Confusion Matrix",
)

if len(test_predictions["true_label"].unique()) == 2:
    fpr, tpr, _ = roc_curve(test_predictions["true_label"], test_predictions["prob_spoof"])
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {test_metrics['roc_auc']:.4f}" if test_metrics["roc_auc"] is not None else "ROC")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{CONFIG['model_name']} - ROC Curve")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(DIRS["curves"] / "roc_curve.png", dpi=160)
    plt.show()
else:
    print("ROC curve skipped because test set does not contain both classes.")


## 12. Show misclassified samples

Save grids of misclassified and correct test examples with true label, prediction, and confidence.


In [ ]:
def plot_prediction_examples(predictions, correct, save_path, title, max_images=16):
    if correct:
        subset = predictions[predictions["is_correct"]].copy()
    else:
        subset = predictions[~predictions["is_correct"]].copy()

    if subset.empty:
        plt.figure(figsize=(8, 3))
        plt.text(0.5, 0.5, "No samples found", ha="center", va="center", fontsize=14)
        plt.axis("off")
        plt.title(title)
        plt.tight_layout()
        plt.savefig(save_path, dpi=160)
        plt.show()
        print(f"No samples for: {title}")
        return

    subset = subset.sample(n=min(max_images, len(subset)), random_state=CONFIG["seed"]).reset_index(drop=True)
    cols = 4
    rows = math.ceil(len(subset) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.0, rows * 4.6))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, subset.iterrows()):
        image = Image.open(row["image_path"]).convert("RGB").resize((CONFIG["image_size"], CONFIG["image_size"]))
        ax.imshow(image)
        ax.axis("off")
        confidence = row["prob_spoof"] if int(row["pred_label"]) == 1 else row["prob_live"]
        sample_title = (
            f"T: {LABEL_MAPPING[int(row['true_label'])]}\n"
            f"P: {LABEL_MAPPING[int(row['pred_label'])]} ({confidence:.2f})"
        )
        if int(row["true_label"]) == 1 and "spoof_type_name" in row:
            sample_title += f"\n{row['spoof_type_name']}"
        ax.set_title(sample_title, fontsize=10)

    for ax in axes[len(subset):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=160)
    plt.show()


plot_prediction_examples(
    test_predictions,
    correct=False,
    save_path=DIRS["samples"] / "misclassified_examples.png",
    title=f"{CONFIG['model_name']} - Misclassified Test Samples",
)
plot_prediction_examples(
    test_predictions,
    correct=True,
    save_path=DIRS["samples"] / "correct_examples.png",
    title=f"{CONFIG['model_name']} - Correct Test Samples",
)


## 13. Final summary

Print the key metrics and output paths in a consistent format for later comparison.


In [ ]:
summary = {
    "Model name": CONFIG["model_name"],
    "Selection metric": "val_acer",
    "Best validation ACER": best_val_acer,
    "Best validation APCER": best_val_apcer,
    "Test Accuracy": test_metrics["accuracy"],
    "Test Precision": test_metrics["precision"],
    "Test Recall": test_metrics["recall"],
    "Test F1": test_metrics["f1"],
    "Test APCER": test_metrics["apcer"],
    "Test BPCER": test_metrics["bpcer"],
    "Test ACER": test_metrics["acer"],
    "Test ROC-AUC": test_metrics["roc_auc"],
    "TN": test_metrics["tn"],
    "FP": test_metrics["fp"],
    "FN": test_metrics["fn"],
    "TP": test_metrics["tp"],
    "Best checkpoint": str(best_checkpoint_path),
    "Output folder": str(OUTPUT_DIR),
}

display(pd.DataFrame([summary]))
print(json.dumps(summary, indent=2, ensure_ascii=False, default=json_default))


This experiment keeps Cross Entropy Loss and GeM Pooling after ResNet18 `layer4`, then adds a projection head with 1-view Supervised Contrastive Loss during training.
